In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

In [2]:
import zipfile

zip_path = "/content/archive (13).zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/oulad')

In [3]:
import os

os.listdir('/content/oulad')

['courses.csv',
 'studentAssessment.csv',
 'vle.csv',
 'studentRegistration.csv',
 'studentVle.csv',
 'assessments.csv',
 'studentInfo.csv']

In [4]:
import pandas as pd

studentInfo = pd.read_csv('/content/oulad/studentInfo.csv')

print(studentInfo.shape)

studentInfo.head()

(32593, 12)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass


In [5]:
studentInfo.columns

Index(['code_module', 'code_presentation', 'id_student', 'gender', 'region',
       'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts',
       'studied_credits', 'disability', 'final_result'],
      dtype='object')

In [6]:
studentInfo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   code_module           32593 non-null  object
 1   code_presentation     32593 non-null  object
 2   id_student            32593 non-null  int64 
 3   gender                32593 non-null  object
 4   region                32593 non-null  object
 5   highest_education     32593 non-null  object
 6   imd_band              31482 non-null  object
 7   age_band              32593 non-null  object
 8   num_of_prev_attempts  32593 non-null  int64 
 9   studied_credits       32593 non-null  int64 
 10  disability            32593 non-null  object
 11  final_result          32593 non-null  object
dtypes: int64(3), object(9)
memory usage: 3.0+ MB


In [7]:
studentInfo['final_result'].value_counts()

,count
final_result,
Pass,12361
Withdrawn,10156
Fail,7052
Distinction,3024


In [8]:
import pandas as pd

studentAssessment = pd.read_csv('/content/oulad/studentAssessment.csv')

print(studentAssessment.shape)

print(studentAssessment.columns)

studentAssessment.head()

(173912, 5)
Index(['id_assessment', 'id_student', 'date_submitted', 'is_banked', 'score'], dtype='object')


,id_assessment,id_student,date_submitted,is_banked,score
0,1752,11391,18,0,78.0
1,1752,28400,22,0,70.0
2,1752,31604,17,0,72.0
3,1752,32885,26,0,69.0
4,1752,38053,19,0,79.0


In [9]:
assessment_features = studentAssessment.groupby('id_student').agg({
    'score':['mean','max','min','count']
})

assessment_features.head()

score                   
                 mean    max   min count
id_student                              
6516        61.800000   77.0  48.0     5
8462        87.000000   93.0  83.0     7
11391       82.000000   85.0  78.0     5
23629       82.500000  100.0  63.0     4
23698       74.444444   94.0  56.0     9

In [10]:
assessment_features.columns = [
    'avg_score',
    'max_score',
    'min_score',
    'assessment_count'
]

assessment_features.reset_index(inplace=True)

assessment_features.head()

,id_student,avg_score,max_score,min_score,assessment_count
0,6516,61.800000,77.0,48.0,5
1,8462,87.000000,93.0,83.0,7
2,11391,82.000000,85.0,78.0,5
3,23629,82.500000,100.0,63.0,4
4,23698,74.444444,94.0,56.0,9


In [11]:
final_df = studentInfo.merge(
    assessment_features,
    on='id_student',
    how='left'
)

final_df.head()

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,avg_score,max_score,min_score,assessment_count
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,82.0,85.0,78.0,5.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,66.4,70.0,60.0,5.0
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn,NaN,NaN,NaN,NaN
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass,76.0,88.0,71.0,5.0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass,54.4,75.0,30.0,5.0


In [12]:
print(final_df.shape)

(32593, 16)


In [13]:
final_df.isnull().sum()

,0
code_module,0
code_presentation,0
id_student,0
gender,0
region,0
highest_education,0
imd_band,1111
age_band,0
num_of_prev_attempts,0
studied_credits,0


In [14]:
final_df['avg_score'] = final_df['avg_score'].fillna(0)
final_df['max_score'] = final_df['max_score'].fillna(0)
final_df['min_score'] = final_df['min_score'].fillna(0)
final_df['assessment_count'] = final_df['assessment_count'].fillna(0)

In [15]:
final_df['imd_band'] = final_df['imd_band'].fillna(
    final_df['imd_band'].mode()[0]
)

In [16]:
studentVle = pd.read_csv(
    '/content/oulad/studentVle.csv'
)

print(studentVle.shape)

print(studentVle.columns)

studentVle.head()

(10655280, 6)
Index(['code_module', 'code_presentation', 'id_student', 'id_site', 'date',
       'sum_click'],
      dtype='object')


,code_module,code_presentation,id_student,id_site,date,sum_click
0,AAA,2013J,28400,546652,-10,4
1,AAA,2013J,28400,546652,-10,1
2,AAA,2013J,28400,546652,-10,1
3,AAA,2013J,28400,546614,-10,11
4,AAA,2013J,28400,546714,-10,1


In [17]:
vle_features = studentVle.groupby('id_student').agg({
    'sum_click':['sum','mean','max','count']
})

vle_features.columns = [
    'total_clicks',
    'avg_clicks',
    'max_clicks',
    'activity_count'
]

vle_features.reset_index(inplace=True)

vle_features.head()

,id_student,total_clicks,avg_clicks,max_clicks,activity_count
0,6516,2791,4.216012,49,662
1,8462,656,2.157895,16,304
2,11391,934,4.765306,76,196
3,23629,161,2.728814,13,59
4,23698,910,2.983607,78,305


In [18]:
final_df = final_df.merge(
    vle_features,
    on='id_student',
    how='left'
)

In [19]:
print(final_df.shape)

(32593, 20)


In [20]:
final_df.isnull().sum()

,0
code_module,0
code_presentation,0
id_student,0
gender,0
region,0
highest_education,0
imd_band,0
age_band,0
num_of_prev_attempts,0
studied_credits,0


In [21]:
final_df['total_clicks'] = final_df['total_clicks'].fillna(0)
final_df['avg_clicks'] = final_df['avg_clicks'].fillna(0)
final_df['max_clicks'] = final_df['max_clicks'].fillna(0)
final_df['activity_count'] = final_df['activity_count'].fillna(0)

In [22]:
click_cols = [
    'total_clicks',
    'avg_clicks',
    'max_clicks',
    'activity_count'
]

final_df[click_cols] = final_df[click_cols].fillna(0)

In [23]:
final_df.to_csv(
    "oulad_final_dataset.csv",
    index=False
)

In [24]:
X = final_df.drop(
    columns=['final_result']
)

y = final_df['final_result']

In [25]:
from sklearn.preprocessing import LabelEncoder

for col in X.select_dtypes(
    include='object'
).columns:

    le = LabelEncoder()

    X[col] = le.fit_transform(X[col])

In [26]:
target_encoder = LabelEncoder()

y = target_encoder.fit_transform(y)

In [27]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [28]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    max_iter=2000
)

lr.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=2000)

In [29]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

RandomForestClassifier(n_estimators=200, random_state=42)

In [30]:
from sklearn.metrics import accuracy_score

pred = rf.predict(X_test)

print(
    accuracy_score(
        y_test,
        pred
    )
)

0.6836938180702562


In [31]:
from sklearn.metrics import classification_report

pred = rf.predict(X_test)

print(classification_report(y_test,pred))

              precision    recall  f1-score   support

           0       0.65      0.48      0.55       605
           1       0.54      0.37      0.44      1411
           2       0.71      0.88      0.79      2472
           3       0.72      0.73      0.72      2031

    accuracy                           0.68      6519
   macro avg       0.65      0.61      0.62      6519
weighted avg       0.67      0.68      0.67      6519



In [32]:
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
})

importance = importance.sort_values(
    by='importance',
    ascending=False
)

print(importance.head(20))

                 feature  importance
14      assessment_count    0.144766
11             avg_score    0.111032
18        activity_count    0.081740
12             max_score    0.080271
15          total_clicks    0.077327
13             min_score    0.076833
2             id_student    0.064684
16            avg_clicks    0.060187
17            max_clicks    0.059657
0            code_module    0.047307
4                 region    0.040559
6               imd_band    0.037508
1      code_presentation    0.031063
9        studied_credits    0.029265
5      highest_education    0.017927
8   num_of_prev_attempts    0.012950
7               age_band    0.010219
3                 gender    0.010039
10            disability    0.006667


In [33]:
final_df = final_df.drop(columns=['id_student'])

In [34]:
final_df['score_range'] = (
    final_df['max_score']
    - final_df['min_score']
)

final_df['engagement_ratio'] = (
    final_df['total_clicks']
    /
    (final_df['assessment_count'] + 1)
)

final_df['performance_index'] = (
    final_df['avg_score']
    *
    final_df['assessment_count']
)

In [35]:
X = final_df.drop(columns=['final_result'])

y = final_df['final_result']

In [36]:
from sklearn.preprocessing import LabelEncoder

X = X.copy()

for col in X.select_dtypes(include='object').columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

In [37]:
target_encoder = LabelEncoder()

y = target_encoder.fit_transform(y)

In [38]:
print(dict(
    zip(
        target_encoder.classes_,
        target_encoder.transform(target_encoder.classes_)
    )
))

{'Distinction': np.int64(0), 'Fail': np.int64(1), 'Pass': np.int64(2), 'Withdrawn': np.int64(3)}


In [39]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [40]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=300, n_jobs=-1,
                       random_state=42)

In [41]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

rf_pred = rf.predict(X_test)

print("RF Accuracy:",
      accuracy_score(y_test, rf_pred))

print(classification_report(
    y_test,
    rf_pred
))

RF Accuracy: 0.681239453903973
              precision    recall  f1-score   support

           0       0.64      0.45      0.53       605
           1       0.55      0.38      0.45      1411
           2       0.71      0.87      0.78      2472
           3       0.71      0.73      0.72      2031

    accuracy                           0.68      6519
   macro avg       0.65      0.61      0.62      6519
weighted avg       0.67      0.68      0.67      6519



In [42]:
import pandas as pd

importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
})

importance = importance.sort_values(
    by='importance',
    ascending=False
)

print(importance.head(20))

                 feature  importance
10             avg_score    0.122501
20     performance_index    0.114063
13      assessment_count    0.070750
12             min_score    0.066662
17        activity_count    0.065261
14          total_clicks    0.062316
18           score_range    0.057041
11             max_score    0.056189
15            avg_clicks    0.053060
19      engagement_ratio    0.051693
16            max_clicks    0.050454
0            code_module    0.043832
3                 region    0.038950
5               imd_band    0.035578
1      code_presentation    0.031196
8        studied_credits    0.027687
4      highest_education    0.016436
7   num_of_prev_attempts    0.012302
6               age_band    0.009198
2                 gender    0.008940


In [43]:
!pip install xgboost -q


In [44]:
from xgboost import XGBClassifier

In [45]:
xgb = XGBClassifier(
    n_estimators=400,
    max_depth=8,
    learning_rate=0.05,
    objective='multi:softprob',
    random_state=42,
    eval_metric='mlogloss'
)

xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=8, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=400, n_jobs=None,
              num_parallel_tree=None, ...)

In [46]:
xgb_pred = xgb.predict(X_test)

print("XGB Accuracy:",
      accuracy_score(y_test, xgb_pred))

print(classification_report(
    y_test,
    xgb_pred
))

XGB Accuracy: 0.6965792299432428
              precision    recall  f1-score   support

           0       0.64      0.48      0.55       605
           1       0.55      0.40      0.46      1411
           2       0.74      0.87      0.80      2472
           3       0.72      0.75      0.74      2031

    accuracy                           0.70      6519
   macro avg       0.66      0.63      0.64      6519
weighted avg       0.68      0.70      0.68      6519



In [47]:
import joblib

joblib.dump(
    xgb,
    'oulad_xgboost.pkl'
)

['oulad_xgboost.pkl']

In [48]:
joblib.dump(
    target_encoder,
    'oulad_target_encoder.pkl'
)

['oulad_target_encoder.pkl']

In [49]:
feature_columns = X.columns.tolist()

joblib.dump(
    feature_columns,
    'oulad_features.pkl'
)

['oulad_features.pkl']

In [50]:
import os

os.listdir()

['.config',
 'oulad_xgboost.pkl',
 'oulad_target_encoder.pkl',
 '.ipynb_checkpoints',
 'oulad',
 'studentVle.csv',
 'oulad_features.pkl',
 'archive (13).zip',
 'oulad_final_dataset.csv',
 'sample_data']

In [51]:
from google.colab import files

files.download('oulad_xgboost.pkl')
files.download('oulad_target_encoder.pkl')
files.download('oulad_features.pkl')
files.download('oulad_final_dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>